# Pancreatic cancer recurrence pipeline on Colab

This notebook clones the repository to Colab's local disk, installs its pinned environment, keeps data/results/checkpoints in Google Drive, and creates a resumable SPECTRE image-embedding run.

Before running: choose **Runtime → Change runtime type → GPU**. The notebook creates `MyDrive/pc-recurrence-prediction/images` and `MyDrive/pc-recurrence-prediction/table`. Upload the *contents* of the curated `dicom_selected` folder into `images/`, and upload the one Excel workbook into `table/`. Patient data stays out of Git, but you should still confirm that using Colab complies with your data-handling requirements.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Edit these if you use a fork, tag/commit, or different Drive folder.
from pathlib import Path

REPO_URL = "https://github.com/sezeriper/pc-recurrence-prediction.git"
REPO_REF = "master"
LOCAL_REPO = Path("/content/pc-recurrence-prediction")
DRIVE_PROJECT = Path("/content/drive/MyDrive/pc-recurrence-prediction")
IMAGES_DIR = DRIVE_PROJECT / "images"
TABLE_DIR = DRIVE_PROJECT / "table"
SPECTRE_RUN = DRIVE_PROJECT / "outputs/image_embeddings/spectre-colab"
for directory in (IMAGES_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Images folder: {IMAGES_DIR}")
print(f"Table folder:  {TABLE_DIR}")

Images folder: /content/drive/MyDrive/pc-recurrence-prediction/images
Table folder:  /content/drive/MyDrive/pc-recurrence-prediction/table


In [ ]:
# Validate Drive inputs. If either is missing, upload it in Google Drive, then
# return here and press Enter. images/ must contain the *contents* of the
# curated dicom_selected directory, not another enclosing dicom_selected folder.
def data_status():
    workbooks = sorted(TABLE_DIR.rglob("*.xlsx"))
    image_files = [
        path
        for path in IMAGES_DIR.rglob("*")
        if path.is_file() and path.name != "curation_manifest.json"
    ]
    manifest = IMAGES_DIR / "curation_manifest.json"
    return workbooks, image_files, manifest


workbooks, image_files, manifest = data_status()
if len(workbooks) != 1 or not image_files or not manifest.is_file():
    print("Data upload required:")
    if not image_files:
        print(f"  1. Upload the contents of curated dicom_selected/ to {IMAGES_DIR}")
    if not manifest.is_file():
        print(f"  2. Copy curation_manifest.json to {IMAGES_DIR}.")
    if len(workbooks) != 1:
        print(f"  3. Upload exactly one .xlsx workbook to {TABLE_DIR} (found {len(workbooks)}).")
    input("After the upload finishes in Google Drive, press Enter to recheck. ")
    workbooks, image_files, manifest = data_status()

if len(workbooks) != 1 or not image_files or not manifest.is_file():
    raise FileNotFoundError(
        f"Expected one .xlsx file in {TABLE_DIR}, DICOM files in {IMAGES_DIR}, and {manifest}."
    )
WORKBOOK = workbooks[0]
CURATED_DICOM = IMAGES_DIR
print(f"Using workbook: {WORKBOOK}")
print(f"Found {len(image_files)} image file(s) in: {CURATED_DICOM}")

Using workbook: /content/drive/MyDrive/pc-recurrence-prediction/table/pankreas adeno ca 10 hasta.xlsx
Found 2777 image file(s) in: /content/drive/MyDrive/pc-recurrence-prediction/images


## Clone and install
The URL must be readable without credentials. For a private repository, use your preferred GitHub authentication method and never paste a token into a shared notebook or commit it.

In [ ]:
import shutil
import subprocess
import sys


def run(*args, cwd=None):
    args = [str(x) for x in args]
    print("+", " ".join(args))
    completed = subprocess.run(
        args, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    if completed.stdout:
        print(completed.stdout, end="" if completed.stdout.endswith("\n") else "\n")
    completed.check_returncode()


if (LOCAL_REPO / ".git").is_dir():
    run("git", "fetch", "--all", "--tags", "--prune", cwd=LOCAL_REPO)
else:
    if LOCAL_REPO.exists():
        raise RuntimeError(f"{LOCAL_REPO} exists but is not a Git checkout.")
    run("git", "clone", REPO_URL, LOCAL_REPO)
run("git", "checkout", REPO_REF, cwd=LOCAL_REPO)
branch_result = subprocess.run(
    ["git", "symbolic-ref", "--quiet", "--short", "HEAD"],
    cwd=LOCAL_REPO,
    text=True,
    capture_output=True,
)
branch = branch_result.stdout.strip()
if branch:
    run("git", "pull", "--ff-only", "origin", branch, cwd=LOCAL_REPO)
# Install uv through the active Colab interpreter. This avoids depending on a
# particular shell profile or home-directory install location.
UV = shutil.which("uv")
if UV is None:
    run(sys.executable, "-m", "pip", "install", "--quiet", "uv")
    UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv installation completed but the uv executable was not on PATH.")
# This project pins Python to >=3.12,<3.13; Colab's system Python can differ.
run(UV, "python", "install", "3.12")
run(UV, "sync", "--python", "3.12", "--extra", "imaging", cwd=LOCAL_REPO)

+ git fetch --all --tags --prune
Fetching origin
+ git checkout master
Already on 'master'
Your branch is up to date with 'origin/master'.
+ git pull --ff-only origin master
From https://github.com/sezeriper/pc-recurrence-prediction
 * branch            master     -> FETCH_HEAD
Already up to date.
+ /usr/local/bin/uv python install 3.12
Python 3.12 is already installed
+ /usr/local/bin/uv sync --python 3.12 --extra imaging
Resolved 91 packages in 1ms
Checked 89 packages in 1ms


In [ ]:
# Use fast local disk for code, but Drive for everything expensive or persistent.
def link_to_drive(name, target):
    local = LOCAL_REPO / name
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() and local.resolve() == target.resolve():
        return
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        raise RuntimeError(f"{local} exists and is not a symlink.")
    local.symlink_to(target, target_is_directory=True)


for name in ("outputs", ".cache"):
    link_to_drive(name, DRIVE_PROJECT / name)
# Print the curated patient-folder layout. Missing/invalid patients are recorded
# and skipped by default by the SPECTRE command below.
folder_check = """
import sys
from pathlib import Path
from pc_recurrence.image_data.workbook import load_image_workbook

dicom_root = Path(sys.argv[1])
workbook = Path(sys.argv[2])
expected = {row.dicom_folder for row in load_image_workbook(workbook) if row.dicom_folder}
actual = {path.name for path in dicom_root.iterdir() if path.is_dir()}
missing = sorted(expected - actual)
print('Expected curated patient folders:', ', '.join(sorted(expected)))
print('Folders found in images/:', ', '.join(sorted(actual)) or '(none)')
if missing:
    print('Will skip unavailable patient folder(s):', ', '.join(missing))
"""
run(UV, "run", "python", "-c", folder_check, CURATED_DICOM, WORKBOOK, cwd=LOCAL_REPO)
gpu_check = (
    "import torch; "
    "print(torch.__version__); "
    'print("CUDA:", torch.cuda.is_available()); '
    'print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")'
)
run(UV, "run", "python", "-c", gpu_check, cwd=LOCAL_REPO)

+ /usr/local/bin/uv run python -c 
import sys
from pathlib import Path
from pc_recurrence.image_data.workbook import load_image_workbook

dicom_root = Path(sys.argv[1])
workbook = Path(sys.argv[2])
expected = {row.dicom_folder for row in load_image_workbook(workbook) if row.dicom_folder}
actual = {path.name for path in dicom_root.iterdir() if path.is_dir()}
missing = sorted(expected - actual)
print('Expected curated patient folders:', ', '.join(sorted(expected)))
print('Folders found in images/:', ', '.join(sorted(actual)) or '(none)')
if missing:
    raise SystemExit('Missing curated patient folder(s): ' + ', '.join(missing))
 /content/drive/MyDrive/pc-recurrence-prediction/images /content/drive/MyDrive/pc-recurrence-prediction/table/pankreas adeno ca 10 hasta.xlsx
Missing curated patient folder(s): PATIENT4227594
Expected curated patient folders: PATIENT2321275, PATIENT2481647, PATIENT2598080, PATIENT2625090, PATIENT2647442, PATIENT3110212, PATIENT3940389, PATIENT4201780, PATIENT4227

CalledProcessError: Command '['/usr/local/bin/uv', 'run', 'python', '-c', "\nimport sys\nfrom pathlib import Path\nfrom pc_recurrence.image_data.workbook import load_image_workbook\n\ndicom_root = Path(sys.argv[1])\nworkbook = Path(sys.argv[2])\nexpected = {row.dicom_folder for row in load_image_workbook(workbook) if row.dicom_folder}\nactual = {path.name for path in dicom_root.iterdir() if path.is_dir()}\nmissing = sorted(expected - actual)\nprint('Expected curated patient folders:', ', '.join(sorted(expected)))\nprint('Folders found in images/:', ', '.join(sorted(actual)) or '(none)')\nif missing:\n    raise SystemExit('Missing curated patient folder(s): ' + ', '.join(missing))\n", '/content/drive/MyDrive/pc-recurrence-prediction/images', '/content/drive/MyDrive/pc-recurrence-prediction/table/pankreas adeno ca 10 hasta.xlsx']' returned non-zero exit status 1.

## Input data layout
`images/` must directly contain the selected DICOM patient folders — the contents that were inside `outputs/dicom_selected/` after preprocessing. `table/` must contain exactly one `.xlsx` workbook. This notebook does not run the DICOM selection/review workflow.

In [ ]:
# Example expected Drive layout:
# pc-recurrence-prediction/images/<patient-folder>/<selected-dicom-files>
# pc-recurrence-prediction/table/pankreas adeno ca 10 hasta.xlsx

## Generate SPECTRE embeddings (GPU)
The stable Drive-backed run directory plus `--resume` allows a compatible interrupted run to continue. The first run downloads pinned weights to the persistent cache. SPECTRE weights are CC-BY-NC-SA and restricted to non-commercial use.

In [ ]:
run(
    UV,
    "run",
    "pc-image-embed",
    "run",
    "--encoder",
    "spectre",
    "--dicom-root",
    CURATED_DICOM,
    "--workbook",
    WORKBOOK,
    "--run-dir",
    SPECTRE_RUN,
    "--resume",
    cwd=LOCAL_REPO,
)
print("SPECTRE embedding artifacts:", SPECTRE_RUN)

## What this produces
The run directory contains `image_embeddings.npz`, `patch_embeddings.npz`, `embedding_summary.csv`, and `run_manifest.json`. The current `pc-recurrence-classify train` CLI requires both Merlin and SPECTRE embedding runs, so recurrence-head training is deliberately not included in this SPECTRE-only notebook.

## Optional repository verification
Uncomment these commands if you modify code in Colab.

In [ ]:
# run(UV, 'sync', '--extra', 'imaging', '--group', 'dev', cwd=LOCAL_REPO)
# run(UV, 'run', 'ruff', 'check', '.', cwd=LOCAL_REPO)
# run(UV, 'run', 'pytest', cwd=LOCAL_REPO)